# CNN与声音的"图像化"

## 学习目标

- 理解语谱图(spectrogram)：音频可以变成"图像"
- 理解卷积操作在频谱图上的含义
- 能用PyTorch搭建并训练一个CNN
- 完成主线项目第一阶段：语音vs噪声分类


## 1. 音频的三种表示

音频信号可以用三种方式表示：
1. **时域**：振幅随时间变化（波形）
2. **频域**：各频率成分的强度（频谱）
3. **时频域**：频率成分随时间的变化（语谱图）

语谱图是深度学习处理音频最常用的表示方式。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchaudio

plt.rcParams['font.size'] = 12

# 生成模拟信号：440Hz正弦波 + 880Hz正弦波（后半段）
sr = 16000
duration = 1.0
t = np.linspace(0, duration, int(sr * duration), endpoint=False)

signal_low = 0.5 * np.sin(2 * np.pi * 440 * t)    # 低频，全程
signal_high = np.zeros_like(t)
signal_high[len(t)//2:] = 0.3 * np.sin(2 * np.pi * 880 * t[len(t)//2:])  # 高频，后半段
signal = signal_low + signal_high

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 时域
axes[0].plot(t[:2000], signal[:2000])
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Amplitude')
axes[0].set_title('时域：波形')
axes[0].grid(True, alpha=0.3)

# 频域
spectrum = np.abs(np.fft.rfft(signal))
freqs = np.fft.rfftfreq(len(signal), 1/sr)
axes[1].plot(freqs[:2000], spectrum[:2000])
axes[1].set_xlabel('Frequency (Hz)')
axes[2].set_ylabel('Magnitude')
axes[1].set_title('频域：频谱')
axes[1].grid(True, alpha=0.3)

# 时频域（语谱图）
from scipy.signal import spectrogram as scipy_spectrogram
f, t_spec, Sxx = scipy_spectrogram(signal, sr, nperseg=256)
axes[2].pcolormesh(t_spec, f, 10 * np.log10(Sxx + 1e-10), shading='gouraud')
axes[2].set_ylabel('Frequency (Hz)')
axes[2].set_xlabel('Time (s)')
axes[2].set_title('时频域：语谱图')
axes[2].set_ylim(0, 2000)

plt.tight_layout()
plt.show()

### 语谱图：深度学习处理音频的关键

语谱图横轴是时间，纵轴是频率，颜色是能量。

**关键洞察**：语谱图就是一张"图片"！CNN天生擅长处理2D图片的模式识别。

所以处理音频的思路是：
1. 把音频变成语谱图
2. 用CNN处理语谱图

这就像给声音"拍照"，然后用图像识别的方法来分析。

## 2. 语谱图的细节

### 梅尔语谱图(Mel Spectrogram)

梅尔刻度模拟了人耳对频率的感知——低频区域分辨率高，高频区域分辨率低。
这与CI的电极映射有内在联系：CI的低频电极对应更多频率信息。

梅尔语谱图是音频深度学习中最常用的特征表示。

In [ ]:
# 用torchaudio提取梅尔语谱图
import torchaudio.transforms as T

# 生成一段模拟语音信号
sr = 16000
duration = 1.0
t = np.linspace(0, duration, int(sr * duration), endpoint=False)
waveform = torch.FloatTensor(0.3 * np.sin(2*np.pi*300*t) + 0.2 * np.sin(2*np.pi*600*t) + 0.1 * np.random.randn(len(t)))
waveform = waveform.unsqueeze(0)  # [1, num_samples]

# 提取梅尔语谱图
mel_transform = T.MelSpectrogram(
    sample_rate=sr,
    n_fft=512,           # FFT窗口大小
    hop_length=160,      # 帧移
    n_mels=64,           # 梅尔频带数
)

mel_spec = mel_transform(waveform)  # [1, n_mels, time_frames]
mel_spec_db = torchaudio.functional.amplitude_to_DB(mel_spec, top_db=80)

print(f'波形形状: {waveform.shape}')
print(f'梅尔语谱图形状: {mel_spec_db.shape}')
print(f'梅尔频带数: {mel_spec_db.shape[1]}')
print(f'时间帧数: {mel_spec_db.shape[2]}')

# 可视化
plt.figure(figsize=(10, 4))
plt.imshow(mel_spec_db[0].numpy(), aspect='auto', origin='lower', cmap='viridis')
plt.colorbar(label='Magnitude (dB)')
plt.xlabel('Time Frame')
plt.ylabel('Mel Band')
plt.title('Mel Spectrogram')
plt.tight_layout()
plt.show()

### 不同类型声音的语谱图对比

下面展示纯净语音、噪声、带噪语音三种语谱图的对比。
理解这个对比对于后续的语音增强（模块5）至关重要。

In [ ]:
# 生成三种信号并对比
np.random.seed(42)
sr = 16000
duration = 0.5
n_samples = int(sr * duration)
t = np.linspace(0, duration, n_samples, endpoint=False)

# 纯净语音模拟：多个谐波
clean = sum(0.3/k * np.sin(2*np.pi*200*k*t) for k in range(1, 6))
# 噪声
noise = 0.1 * np.random.randn(n_samples)
# 带噪语音
noisy = clean + noise

fig, axes = plt.subplots(3, 2, figsize=(14, 8))

mel_transform = T.MelSpectrogram(sample_rate=sr, n_fft=512, hop_length=160, n_mels=64)

for i, (sig, name) in enumerate([(clean, '纯净语音'), (noise, '噪声'), (noisy, '带噪语音')]):
    # 波形
    axes[i, 0].plot(t[:2000], sig[:2000])
    axes[i, 0].set_ylabel('Amplitude')
    axes[i, 0].set_title(f'{name} - 波形')
    axes[i, 0].grid(True, alpha=0.3)
    
    # 语谱图
    sig_tensor = torch.FloatTensor(sig).unsqueeze(0)
    mel = torchaudio.functional.amplitude_to_DB(mel_transform(sig_tensor), top_db=80)
    axes[i, 1].imshow(mel[0].numpy(), aspect='auto', origin='lower', cmap='viridis')
    axes[i, 1].set_title(f'{name} - 语谱图')
    axes[i, 1].set_ylabel('Mel Band')

axes[2, 0].set_xlabel('Time (s)')
axes[2, 1].set_xlabel('Time Frame')
plt.tight_layout()
plt.show()

print('观察：语音的语谱图有清晰的条纹结构，噪声的语谱图是均匀分布的。')
print('语音增强的目标就是：把带噪语音的语谱图"变干净"——去除均匀的噪声，保留清晰的条纹。')

## 3. 卷积神经网络(CNN)

### 3.1 卷积操作

卷积在语谱图上的直觉：一个小窗口（卷积核）在语谱图上滑动，提取局部时频模式。

- **卷积核** = 一个小的模式检测器
- **步长(stride)** = 每次滑动的距离
- **填充(padding)** = 在边缘补零，保持输出尺寸

In [ ]:
# CNN核心操作演示
import torch.nn.functional as F

# 创建一个简单的8x8语谱图
spec = torch.randn(1, 1, 8, 8)  # [batch, channel, height, width]

# 3x3卷积核
conv = nn.Conv2d(1, 1, kernel_size=3, stride=1, padding=0)

output = conv(spec)
print(f'输入形状: {spec.shape}')
print(f'输出形状: {output.shape}')
print(f'3x3卷积，无填充: 8→6 (8-3+1=6)')

# 加入填充
conv_pad = nn.Conv2d(1, 1, kernel_size=3, stride=1, padding=1)
output_pad = conv_pad(spec)
print(f'\n加入padding=1后输出形状: {output_pad.shape}')
print(f'3x3卷积，padding=1: 8→8 (尺寸不变)')

# 步长为2
conv_stride = nn.Conv2d(1, 1, kernel_size=3, stride=2, padding=1)
output_stride = conv_stride(spec)
print(f'\nstride=2后输出形状: {output_stride.shape}')
print(f'3x3卷积，stride=2，padding=1: 8→4 (尺寸减半)')

### 3.2 池化(Pooling)

池化操作减小特征图的尺寸，保留最重要的信息。

In [ ]:
# 池化演示
x = torch.randn(1, 1, 4, 4)
print('输入:')
print(x[0, 0].numpy())

# 最大池化
maxpool = nn.MaxPool2d(kernel_size=2, stride=2)
y = maxpool(x)
print(f'\n最大池化后形状: {y.shape}')
print('输出:')
print(y[0, 0].numpy())
print('\n每个2x2区域取最大值，尺寸减半')

## 4. 搭建CNN音频分类器

现在我们搭建一个CNN来分类语谱图：**语音 vs 噪声**。

这是主线项目的第一阶段。

In [ ]:
# 生成语音/噪声分类数据集
import torch
from torch.utils.data import Dataset, DataLoader

class SpeechNoiseDataset(Dataset):
    """语音/噪声分类数据集"""
    def __init__(self, n_samples=800, sr=16000, duration=0.5):
        self.sr = sr
        self.duration = duration
        self.n_samples = n_samples
        
        # 预生成数据
        self.data = []
        self.labels = []
        n_samples_signal = int(sr * duration)
        
        for i in range(n_samples):
            t = np.linspace(0, duration, n_samples_signal, endpoint=False)
            
            if i < n_samples // 2:
                # 语音：多个谐波
                freq = np.random.uniform(100, 400)
                signal = sum(np.random.uniform(0.1, 0.4) * np.sin(2*np.pi*freq*k*t) for k in range(1, np.random.randint(3, 7)))
                signal += 0.02 * np.random.randn(n_samples_signal)  # 微弱噪声
                label = 0  # 语音
            else:
                # 噪声：白噪声或粉红噪声
                signal = np.random.uniform(0.05, 0.3) * np.random.randn(n_samples_signal)
                label = 1  # 噪声
            
            self.data.append(torch.FloatTensor(signal))
            self.labels.append(label)
    
    def __len__(self):
        return self.n_samples
    
    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

# 创建数据集
dataset = SpeechNoiseDataset(n_samples=800)
print(f'数据集大小: {len(dataset)}')
print(f'语音样本数: {sum(1 for l in dataset.labels if l == 0)}')
print(f'噪声样本数: {sum(1 for l in dataset.labels if l == 1)}')

# 划分训练/验证集
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])
print(f'训练集: {train_size}, 验证集: {val_size}')

### 搭建CNN模型

你需要自己填写一些关键参数。

In [ ]:
# ====== 练习：搭建CNN ======

class AudioCNN(nn.Module):
    def __init__(self, n_mels=64, n_classes=2):
        super().__init__()
        self.mel_transform = T.MelSpectrogram(
            sample_rate=16000,
            n_fft=512,
            hop_length=160,
            n_mels=n_mels,
        )
        
        # TODO: 定义卷积层
        # 提示：输入是 [batch, 1, n_mels, time_frames]
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),   # [B, 16, n_mels, T]
            nn.ReLU(),
            nn.MaxPool2d(2),                               # [B, 16, n_mels/2, T/2]
            nn.Conv2d(16, 32, kernel_size=3, padding=1),  # [B, 32, n_mels/2, T/2]
            nn.ReLU(),
            nn.MaxPool2d(2),                               # [B, 32, n_mels/4, T/4]
            nn.Conv2d(32, 64, kernel_size=3, padding=1),  # [B, 64, n_mels/4, T/4]
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),                  # [B, 64, 1, 1] 全局平均池化
        )
        
        # TODO: 定义分类器
        self.classifier = nn.Linear(64, n_classes)
    
    def forward(self, x):
        # x: [batch, num_samples]
        # 提取梅尔语谱图
        mel = self.mel_transform(x)            # [B, 1, n_mels, T]
        mel_db = torchaudio.functional.amplitude_to_DB(mel, top_db=80)
        mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-8)  # 归一化
        mel_db = mel_db.unsqueeze(1)           # [B, 1, n_mels, T] 添加通道维
        
        # 卷积特征提取
        features = self.features(mel_db)       # [B, 64, 1, 1]
        features = features.view(features.size(0), -1)  # [B, 64]
        
        # 分类
        output = self.classifier(features)     # [B, n_classes]
        return output

# 测试模型
model = AudioCNN()
test_input = torch.randn(4, 8000)  # batch=4, 0.5秒音频
test_output = model(test_input)
print(f'输入形状: {test_input.shape}')
print(f'输出形状: {test_output.shape}')
print(f'模型参数总数: {sum(p.numel() for p in model.parameters()):,}')

### 训练CNN

现在训练这个CNN分类器。这次你要自己写训练循环的部分代码。

In [ ]:
# ====== 练习：训练CNN ======

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = AudioCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

# 训练
n_epochs = 20
train_losses = []
val_accs = []

for epoch in range(n_epochs):
    model.train()
    epoch_loss = 0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        # TODO: 完成训练循环
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    train_losses.append(epoch_loss / len(train_loader))
    
    # 验证
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            outputs = model(batch_x)
            _, predicted = outputs.max(1)
            total += batch_y.size(0)
            correct += predicted.eq(batch_y).sum().item()
    
    val_acc = correct / total
    val_accs.append(val_acc)
    
    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1:2d}, Loss: {train_losses[-1]:.4f}, Val Acc: {val_acc:.4f}')

print(f'\n最终验证准确率: {val_accs[-1]:.4f}')

In [ ]:
# 可视化训练过程
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(train_losses)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(val_accs)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Validation Accuracy')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 挑战任务

1. 修改CNN结构：增加/减少卷积层，观察效果
2. 修改 `n_mels`（32, 64, 128），观察对准确率和训练速度的影响
3. 用 `nn.Sequential` 替换手动定义的 `features`，比较代码可读性
4. 增加一个 `nn.Dropout(0.5)` 层在分类器之前，观察是否缓解过拟合

## 本节要点

1. **语谱图**是音频深度学习的核心表示——横轴时间、纵轴频率、颜色能量
2. **梅尔语谱图**模拟人耳感知——与CI电极映射有内在联系
3. **CNN处理音频 = CNN处理语谱图"图片"**
4. **卷积**= 模式检测器在语谱图上滑动，寻找局部时频模式
5. **池化**= 减小特征图尺寸，保留重要信息
6. 训练CNN的完整流程：数据 → 特征提取 → 模型 → 训练 → 评估


---
下一节：[04-training-tricks.ipynb](04-training-tricks.ipynb) — 训练技巧与实验方法论
